# VSE-16g | Incubator Hotel Vibration & Labware Pattern Screening

**Engineering analysis of raw three-axis Vibration Sentry E-16g acceleration.** This separate notebook does not change the Temperature/RH analytics notebook.

**Workflow:** Install → mount Google Drive (optional) → enter raw file and labware-model paths → **Load** → select time / sample window → **Analyze** → inspect time RMS, Welch PSD, FFT, spectrogram, modal ranking and short-time labware-mode exposure → export.

**Not a proof of cell pattern formation:** The instrument is mounted on the hotel, not at the culture fluid. The labware threshold definition (peak vs RMS, drive axis, experimental validation) is not supplied. The labware file's four-hour stabilization period is longer than the uploaded 58-minute example. Treat all matching/ratio results as *screening hypotheses* only.

This code handles text exports that end in **`.xls` but are really tab-delimited text**, and also genuine Excel when the required reader is available. Upload real experimental inputs only into your private Drive or the temporary Colab session; **do not commit them to public GitHub**.

### 1. Install analysis dependencies

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "scipy>=1.11,<2" "plotly>=5.20,<7" "ipywidgets>=8,<9" "xlrd>=2,<3" "openpyxl>=3.1,<4"

### 2. Load the current engine from GitHub and enable Colab controls

In [ ]:
from pathlib import Path
import os, sys, re, json, importlib
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import display, HTML, clear_output
from urllib.request import urlretrieve
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    print('Colab widget communication enabled.')
except ImportError:
    print('Running outside Google Colab.')
URL='https://raw.githubusercontent.com/leotirto/Cell-Incubator-Chamber-Temperature-RH-Analytics/main/vibration_analysis/vse_vibration_core.py'
ENGINE=Path('vse_vibration_core.py')
print('Refreshing analysis engine from GitHub...')
urlretrieve(URL,str(ENGINE))
if 'vse_vibration_core' in sys.modules:
    importlib.reload(sys.modules['vse_vibration_core'])
from vse_vibration_core import (
   read_vse, read_labware_reference, analyze, spectrogram, modal_time_exposure,
   modal_time_summary, plot_raw_waveform, plot_time_rms, plot_time_shocks,
   plot_time_sections, plot_psd, plot_fft, plot_spectrogram,plot_mode_ranking,
   plot_modal_time_exposure, export_report, SOURCE_URLS
)
print('Analysis engine ready. Enter raw recording and labware reference paths below.')

### 3. Dashboard — paths, time window, signal processing, and plots

The timestamp and sample-range filters are **always visible**, not hidden inside a collapsed menu. Date/time is interpreted in the logger's local clock. Blank values mean no restriction; row/time and elapsed-seconds filters intersect. In this notebook **Sample 1** means the *first acceleration reading* (not the header or metadata lines).

In [ ]:
# LOAD INPUTS
raw_file=w.Text(value='',description='Raw VSE file:',placeholder='/content/drive/MyDrive/.../your_export.xls',
                layout=w.Layout(width='850px'),style={'description_width':'140px'})
ref_file=w.Text(value='',description='Labware TXT:',placeholder='/content/drive/MyDrive/.../Frequency Affect to Cell Pattern.txt',
                layout=w.Layout(width='850px'),style={'description_width':'140px'})
load_button=w.Button(description='Load recording + reference',button_style='info',icon='folder-open',
                     layout=w.Layout(width='270px'))
load_out=w.Output(layout=w.Layout(min_height='50px'))
loaded_info=w.HTML(value='<small>No file loaded yet. After mounting Drive, enter exact file paths.</small>')

# ALWAYS VISIBLE TIME / INDEX FILTERS
from_time=w.Text(value='',description='From time:',placeholder='YYYY-MM-DD HH:MM:SS.ffffff',
                 layout=w.Layout(width='480px'),style={'description_width':'120px'})
to_time=w.Text(value='',description='To time:',placeholder='YYYY-MM-DD HH:MM:SS.ffffff',
               layout=w.Layout(width='480px'),style={'description_width':'120px'})
from_s=w.FloatText(value=0,description='Start elapsed (s):',layout=w.Layout(width='280px'),
                   style={'description_width':'155px'})
to_s=w.FloatText(value=0,description='End elapsed (s):',layout=w.Layout(width='280px'),
                 style={'description_width':'155px'})
from_sample=w.IntText(value=0,description='Start sample:',layout=w.Layout(width='280px'),
                      style={'description_width':'155px'})
to_sample=w.IntText(value=0,description='End sample:',layout=w.Layout(width='280px'),
                    style={'description_width':'155px'})
all_button=w.Button(description='Full recording',icon='maximize',layout=w.Layout(width='180px'))
skip_button=w.Button(description='Skip first 5 minutes',icon='forward',layout=w.Layout(width='205px'))
first_button=w.Button(description='First 5 minutes',icon='clock-o',layout=w.Layout(width='175px'))
last_button=w.Button(description='Last 10 minutes',icon='clock-o',layout=w.Layout(width='175px'))
window_info=w.HTML(value='<small>Zero elapsed seconds / sample numbers = no restriction.</small>')

# ANALYSIS CONTROLS
axis=w.Dropdown(options=['Vector','X','Y','Z'],value='Vector',description='Spectrum axis:',style={'description_width':'125px'})
labware=w.Dropdown(options=['All'],value='All',description='Labware:',layout=w.Layout(width='650px'),
                   style={'description_width':'125px'})
welch_seconds=w.Dropdown(options=[('10.24 s / 0.098 Hz',10.24),('20.48 s / 0.049 Hz',20.48),
                                   ('40.96 s / 0.024 Hz',40.96)],value=20.48,
                         description='Welch window:',style={'description_width':'125px'})
rms_bin=w.Dropdown(options=[1,2,5,10],value=1,description='RMS bin (s):',style={'description_width':'125px'})
f_min=w.FloatText(value=1,description='Min Hz:',style={'description_width':'125px'})
f_max=w.FloatText(value=300,description='Max Hz:',style={'description_width':'125px'})
band_halfwidth=w.FloatText(value=.5,description='Mode ±Hz min:',style={'description_width':'125px'})
mode_count=w.IntSlider(value=8,min=1,max=24,description='Top modes:',style={'description_width':'125px'})
fft_snap=w.Dropdown(options=[10,30,60],value=30,description='Snapshot FFT (s):',style={'description_width':'125px'})
hp_cutoff=w.FloatText(value=.5,description='RMS highpass Hz:',style={'description_width':'140px'})

analyze_button=w.Button(description='Analyze selected window',icon='chart-bar',button_style='success',
                       layout=w.Layout(width='250px'))
export_button=w.Button(description='Export CSV + interactive HTML ZIP',icon='download',button_style='warning',
                      layout=w.Layout(width='300px'))
analysis_feedback=w.Output(layout=w.Layout(min_height='35px'))
sections=['Overview & QA','Vibration vs time','Welch PSD + FFT','Time-frequency heatmap',
          'Labware modes','Time exposure','Tables & export']
outputs={name:w.Output(layout=w.Layout(min_height='140px')) for name in sections}
tabs=w.Tab(children=list(outputs.values()))
for i,key in enumerate(sections):tabs.set_title(i,key)
state={'log':None,'reference':None,'result':None,'figures':{}}
source_panel=w.VBox([w.HTML('<h3>1. Input source</h3>'),raw_file,ref_file,
                      w.HBox([load_button]),loaded_info,load_out])
time_panel=w.VBox([w.HTML('<h3 style="color:#0f766e">2. TIME WINDOW — always visible</h3>'),
                   w.HBox([from_time,to_time]),
                   w.HBox([from_s,to_s,from_sample,to_sample]),
                   w.HBox([all_button,skip_button,first_button,last_button]),window_info])
settings_panel=w.VBox([w.HBox([axis,labware]),w.HBox([welch_seconds,rms_bin,fft_snap]),
                       w.HBox([f_min,f_max,band_halfwidth,hp_cutoff]),mode_count,
                       w.HTML('<small>Reference frequencies are compared to PSD-band RMS at the SENSOR. '
                              'Threshold amplitude convention and hotel-to-well transmission are unknown.</small>')])
settings_accordion=w.Accordion(children=[settings_panel],selected_index=None)
settings_accordion.set_title(0,'3. Advanced spectral and modal settings')
dashboard=w.VBox([w.HTML('<h2>VSE-16g | Hotel vibration analytics</h2>'),
                  source_panel,time_panel,settings_accordion,
                  w.HBox([analyze_button,export_button]),analysis_feedback,tabs])
display(dashboard)

### 4. Activate the dashboard buttons

In [ ]:
def _mount_if_needed(path_text:str):
    path=Path(path_text.strip()).expanduser()
    if str(path).startswith('/content/drive/') and not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    return path

def _load(_=None):
    with load_out:
        clear_output(wait=True)
        try:
            if not raw_file.value.strip():
                print('Choose a raw .xls/.csv file. For native upload use the optional code cell below.')
                return
            path=_mount_if_needed(raw_file.value)
            if not path.exists():raise FileNotFoundError(str(path))
            print('Loading raw samples (large exports can contain millions of readings)...')
            log=read_vse(path)
            reference=None
            if ref_file.value.strip():
                rp=_mount_if_needed(ref_file.value)
                if not rp.exists():raise FileNotFoundError(f'Labware reference: {rp}')
                reference=read_labware_reference(rp)
            state['log']=log
            state['reference']=reference
            state['result']=None
            labware.options=['All']+([] if reference is None else sorted(reference.Labware.unique()))
            labware.value='All'
            q=log.quality
            loaded_info.value=(f'<b>Loaded:</b> {q["rows_valid"]:,} samples | '
              f'{q["fs_est_Hz"]:,.2f} Hz | {q["duration_s"]/60:.2f} min | '
              f'{q["start"]} to {q["end"]} | '
              f'{0 if reference is None else len(reference)} reference modes')
            print(loaded_info.value)
            if reference is None:print('Reference TXT not provided: only vibration physics views will be available.')
        except Exception as exc:
            print('LOAD FAILED:',type(exc).__name__,str(exc))
            state['log']=None

def _preset(kind):
    log=state['log']
    if log is None:
        window_info.value='<b>Load a CSV first.</b>'
        return
    from_time.value='';to_time.value='';from_sample.value=0;to_sample.value=0
    dur=log.quality['duration_s']
    start,end={'all':(0,0),'skip':(300,0),'first':(0,min(300,dur)),
               'last':(max(0,dur-600),0)}[kind]
    from_s.value=start
    to_s.value=end
    window_info.value=f'<small>Filter selected: {kind}, start {start:.2f} s, end {end:.2f} s (0 = no end)</small>'

def _settings():
    if float(f_min.value)>=float(f_max.value):raise ValueError('Max frequency must exceed min frequency')
    if from_s.value<0 or to_s.value<0:raise ValueError('Elapsed time values cannot be negative')
    return dict(start_time=from_time.value.strip() or None,end_time=to_time.value.strip() or None,
                start_s=from_s.value if from_s.value>0 else None,
                end_s=to_s.value if to_s.value>0 else None,
                start_sample=from_sample.value if from_sample.value>0 else None,
                end_sample=to_sample.value if to_sample.value>0 else None,
                welch_s=welch_seconds.value,fft_snapshot_s=fft_snap.value,
                axis=axis.value,rms_bin_s=rms_bin.value,
                highpass_Hz=hp_cutoff.value,labware=labware.value,
                band_halfwidth_Hz=band_halfwidth.value,
                fmin=f_min.value,fmax=f_max.value)

def _analyze(_=None):
    with analysis_feedback:
        clear_output(wait=True)
        if state['log'] is None:
            print('LOAD a raw recording first. See the visible Input source section.')
            return
        print('Processing selected samples, Welch PSD, FFT and short-time exposure...')
    try:
        result=analyze(state['log'],state['reference'],**_settings())
        state['result']=result
        heat=spectrogram(result['selected'],state['log'].quality['fs_est_Hz'],axis=axis.value,
                         fmax=f_max.value)
        exposure=modal_time_exposure(result['selected'],state['log'].quality['fs_est_Hz'],
                                   result['labware_matches'],top_modes=mode_count.value)
        result['modal_exposure']=exposure
        figures={
          '01_raw_waveform':plot_raw_waveform(result,axis=axis.value),
          '02_rms_over_time':plot_time_rms(result),
          '03_shocks_over_time':plot_time_shocks(result),
          '04_five_minute_bars':plot_time_sections(result),
          '05_welch_psd':plot_psd(result,state['reference'],axis=axis.value),
          '06_fft_snapshot':plot_fft(result,axis=axis.value),
          '07_spectrogram':plot_spectrogram(heat),
          '08_modal_ranking':plot_mode_ranking(result),
          '09_modal_exposure':plot_modal_time_exposure(exposure)}
        state['figures']=figures
        qa=result['quality']
        with outputs['Overview & QA']:
            clear_output(wait=True)
            display(HTML('<h3>Measurement-integrity and physical interpretation</h3>'))
            display(pd.DataFrame([qa]).T.rename(columns={0:'Value'}).loc[
             ['file','selected_from','selected_to','selected_samples',
              'selected_duration_s','fs_est_Hz','nyquist_Hz',
              'time_duplicates','sampling_gaps_gt_1_5_period',
              'welch_resolution_Hz','spectral_coverage_pct',
              'median_vector_rms_m_s2','p95_vector_rms_m_s2','max_bin_rms_m_s2',
              'peak_dynamic_sample_m_s2']])
            for message in qa['warnings']:print('CAUTION:',message)
            display(HTML('<h3>Dominant measured spectral peaks</h3>'))
            display(result['peak_table'].head(20).round(4))
        with outputs['Vibration vs time']:
            clear_output(wait=True)
            for k in ['01_raw_waveform','02_rms_over_time','03_shocks_over_time','04_five_minute_bars']:
                figures[k].show()
            display(result['time_sections'].round(4))
        with outputs['Welch PSD + FFT']:
            clear_output(wait=True)
            print('PSD is the primary frequency-energy metric. FFT graph is one centered snapshot.')
            figures['05_welch_psd'].show();figures['06_fft_snapshot'].show()
            display(result['peak_table'].head(25).round(5))
        with outputs['Time-frequency heatmap']:
            clear_output(wait=True)
            print('Spectrogram shows persistent tones versus short vibration events. Unit: (m/s²)²/Hz in decibels.')
            figures['07_spectrogram'].show()
        with outputs['Labware modes']:
            clear_output(wait=True)
            print('SCREENING ONLY — sensor measurements are not plate acceleration. Threshold metric may not mean RMS.')
            if state['reference'] is None:print('Load your TXT labware file to compare supplied frequencies.')
            else:
                figures['08_modal_ranking'].show()
                display(result['labware_matches'].sort_values(
                     'Tonal_excess_to_reference_ratio',ascending=False).head(35).round(4))
                print('Ratio > 1 is NOT a fail criterion or proof of pattern formation.')
        with outputs['Time exposure']:
            clear_output(wait=True)
            if exposure.empty:print('Upload the labware TXT to enable this view.')
            else:
                figures['09_modal_exposure'].show()
                display(modal_time_summary(exposure).round(3))
                print('These are OVERLAPPING FFT windows; fraction above ratio 1 is descriptive, not independent time-at-risk.')
        with outputs['Tables & export']:
            clear_output(wait=True)
            print('Selected-window time bins:');display(result['time_metrics'].head(20).round(4))
            print('Welch spectrum:');display(result['spectrum'].head(15).round(6))
            print('Export button saves all processed tables and interactive HTML charts.')
        tabs.selected_index=0
        with analysis_feedback:
            clear_output(wait=True)
            print(f"Done — {qa['selected_samples']:,} samples, {qa['selected_duration_s']/60:.2f} min; "
                  f"{len(result['labware_matches'])} labware modes screened. Open tabs above.")
    except Exception as exc:
        with analysis_feedback:
            clear_output(wait=True)
            print('ANALYSIS FAILED:',type(exc).__name__,str(exc))

def _export(_=None):
    with analysis_feedback:
        clear_output(wait=True)
        if state['result'] is None:
            print('Analyze a selected time range first.')
            return
        try:
            from zipfile import ZipFile,ZIP_DEFLATED
            dest=Path('/content/vse_analysis_exports') if Path('/content').is_dir() else Path('vse_analysis_exports')
            export_report(state['result'],dest,figures=state['figures'])
            archive=dest.parent/'vse_vibration_report.zip'
            with ZipFile(archive,'w',compression=ZIP_DEFLATED) as z:
                for p in dest.rglob('*'):
                    if p.is_file():z.write(p,arcname=str(p.relative_to(dest.parent)))
            print('Results ZIP:',archive)
            try:
                from google.colab import files
                files.download(str(archive))
            except ImportError:
                print('Open the path shown above.')
        except Exception as exc:
            print('EXPORT FAILED:',type(exc).__name__,str(exc))

load_button.on_click(_load)
analyze_button.on_click(_analyze)
export_button.on_click(_export)
all_button.on_click(lambda _: _preset('all'))
skip_button.on_click(lambda _: _preset('skip'))
first_button.on_click(lambda _: _preset('first'))
last_button.on_click(lambda _: _preset('last'))
print('Buttons are active. Step 1: load files, Step 2: select window, Step 3: analyze.')

### 5. Optional: native Colab file upload (only when manually enabled)

To avoid an unresponsive upload dialog inside widget callbacks, this cell uses Colab's native upload interface **directly**. Change `UPLOAD_NOW` to `True` and execute **this cell only**. Its default `False` ensures `Runtime → Run all` does not unexpectedly open a file chooser. You may upload both the raw `.xls`/`.csv` and the small `.txt` labware reference at the same time.

In [ ]:
UPLOAD_NOW = False
if UPLOAD_NOW:
    from google.colab import files
    folder=Path('/content/vse_private_uploads')
    folder.mkdir(parents=True,exist_ok=True)
    uploaded=files.upload()
    for name,payload in uploaded.items():
        safe=Path(name).name
        (folder/safe).write_bytes(payload)
        if safe.lower().endswith(('.xls','.xlsx','.csv')):
            raw_file.value=str(folder/safe)
        elif safe.lower().endswith('.txt'):
            ref_file.value=str(folder/safe)
        print('Received:',safe)
    print('Now click Load recording + reference in the dashboard.')
else:
    print('Upload is optional. To use it, set UPLOAD_NOW = True and run this cell.')

### 6. Direct, reproducible code-cell fallback (without buttons)

If Colab cannot communicate with an `ipywidgets` button, edit the file paths under **Input source** and optionally the time filters above, then set `RUN_DIRECT_ANALYSIS = True` and execute this cell. It uses the same window settings and produces charts without widget callbacks. Default `False` is safe for Run all.

In [ ]:
RUN_DIRECT_ANALYSIS = False
if RUN_DIRECT_ANALYSIS:
    rawpath=Path(raw_file.value.strip())
    if not rawpath.is_file():
        raise FileNotFoundError(f'Set a valid raw file path first: {rawpath}')
    logfile=read_vse(rawpath)
    reference=read_labware_reference(ref_file.value) if ref_file.value.strip() else None
    options=_settings()
    report=analyze(logfile,reference,**options)
    heat=spectrogram(report['selected'],logfile.quality['fs_est_Hz'],axis=axis.value,fmax=f_max.value)
    exp=modal_time_exposure(report['selected'],logfile.quality['fs_est_Hz'],report['labware_matches'],top_modes=mode_count.value)
    report['modal_exposure']=exp
    display(pd.DataFrame([report['quality']]).T)
    for chart in [plot_time_rms(report),plot_time_sections(report),plot_psd(report,reference,axis=axis.value),
                  plot_fft(report,axis=axis.value),plot_spectrogram(heat),
                  plot_mode_ranking(report),plot_modal_time_exposure(exp)]:
        chart.show()
    display(report['peak_table'].head(25).round(4))
    display(report['labware_matches'].sort_values('Tonal_excess_to_reference_ratio',ascending=False).head(25).round(4))
else:
    print('Direct analysis is optional. Set RUN_DIRECT_ANALYSIS = True and rerun if button callbacks fail.')

### 7. Engineering interpretation & critical assumptions

1. **Sampling and identifiable range:** At 800 Hz, the 400 Hz Nyquist bound covers the supplied 1.2–295 Hz model modes. This does not establish sufficient sensor response or eliminate aliasing; consult the VSE manual and configured high-pass settings.
2. **Gravity versus vibration:** Raw acceleration includes orientation/gravity. High-pass-filtered time-domain RMS removes low-frequency drift (adjustable cutoff 0.5 Hz), while Welch PSD removes only the sample mean per Hann segment. These distinct processing choices are documented in exports.
3. **FFT/PSD:** A single FFT shows narrow tones for one snapshot. Welch PSD estimates mean-square acceleration density over a longer interval; PSD frequency-bin integration estimates band **RMS**. The one-sided vector PSD sums three axis powers and does not model phase-dependent well motion.
4. **Reference comparison:** `Frequency Affect to Cell Pattern.txt` lists specific labware/harmonic frequencies, spacing, and an acceleration threshold with a four-hour stabilization window. It does **not** define whether the thresholds are RMS or peak, measurement axis/location, transfer functions, or validated experimental outcomes. Ratios >1 are **screening only**.
5. **Physical coupling:** Hotel → shelf → labware → fluid transmission is frequency-dependent. To claim that a measured frequency creates a cell pattern requires measurements near the well, media height, attachment stage and cell imaging. Faraday-like subharmonic responses could produce fluid patterns at half the drive frequency; this notebook does not assume those are the same as reference harmonics.
6. **Instrument floor:** Manufacturer typical 800 Hz noise is about 4.5 mg RMS on X/Y and 8.9 mg RMS on Z for the E-16g. Sub-mg reference thresholds require careful narrowband uncertainty analysis and calibration; instantaneous peaks and PSD-power estimates are not the same as a calibrated fluid-surface response.
7. **Transient bias:** A shock/handling period can dominate the whole-record spectrum. Compare full duration against **Skip first 5 minutes**; report both instead of treating one as a universal operating condition.

**External physics/engineering references:** [VSE-16g product](https://convergenceinstruments.com/product/vibration-meter-datalogger-vibration-sentry-e16g/), [manufacturer specifications](https://convergenceinstruments.com/pdf/vibration_sentry_e_datasheet.pdf), [cell-culture vessel pattern technical note](https://documents.thermofisher.com/TFS-Assets/BID/Technical-Notes/vibration-patterns-cell-culture-vessels-technical-note.pdf), [peer-reviewed review of acoustic cell patterning](https://pmc.ncbi.nlm.nih.gov/articles/PMC8094912/).